# Warpmate, Phase 3: the neural network

This notebook:

1. Builds Warpmate and runs the tests (including two-of-a-kind pieces like KQQvK).
2. **Solves every 4-piece endgame** except KPvKP (29) plus the 3-piece ones, on the GPU.
3. Checks every table against the official **Syzygy** tablebases.
4. **Builds the dataset** from the tables and spot-checks its labels against Syzygy.
5. **Trains the network** (769 → 256 → 64 → 3) with PyTorch on the GPU.
6. **Evaluates it**: test accuracy, move choice, and 5-piece endgames it never saw.

About 20 to 25 minutes in total. **Before you start:** *Runtime > Change runtime type > T4 GPU > Save*, then run the cells top to bottom.

In [ ]:
!nvidia-smi

## Get the code

If `warpmate-colab.zip` is in the **Files** panel on the left (drag it there), it's used; otherwise the code is cloned from GitHub.

In [ ]:
import os
os.chdir('/content')
if os.path.exists('/content/warpmate-colab.zip'):
    !rm -rf Warpmate && mkdir Warpmate && unzip -q warpmate-colab.zip -d Warpmate
    print('Using the uploaded zip')
else:
    !rm -rf Warpmate && git clone -q https://github.com/vasishtk1/Warpmate.git
    print('Cloned from GitHub')
%cd /content/Warpmate
!test -f ml/train.py && echo "Phase 3 code present" || echo "WARNING: this copy doesn't have the Phase 3 code yet"

## Build

Look for `Warpmate GPU support: ON`.

In [ ]:
import os
if os.path.exists('/usr/local/cuda/bin/nvcc'):
    os.environ['CUDACXX'] = '/usr/local/cuda/bin/nvcc'
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -4
!cmake --build build -j 2>&1 | tail -4

## Tests

`egtb-gpu` now also solves KQQvK, KBBvK and KNNvK and checks that swapping the two identical pieces never changes a value.

In [ ]:
!ctest --test-dir build --output-on-failure

## Solve every 3- and 4-piece endgame on the GPU

34 tables (5 three-piece, 29 four-piece), about 33.5 MB each. Should take under a minute.

In [ ]:
!./build/warpmate-gen --gpu --all3 --all4 2>&1 | tee gen_all.log | grep -v 'longest win'

## Check every table against Syzygy

Downloads the official win/draw/loss files for all 3- and 4-piece endgames (KPvKP too: Syzygy needs it for some lookups), then checks a random sample of 200,000 rows of every table. About 10 to 12 minutes.

In [ ]:
!pip -q install chess
!mkdir -p syzygy
import glob, os, subprocess
names = sorted(os.path.basename(p)[:-5] for p in glob.glob('tables/*.wmtb')) + ['KPvKP']
for t in names:
    subprocess.run(['wget', '-q', '-nc', '-P', 'syzygy',
                    f'https://tablebase.lichess.ovh/tables/standard/3-4-5-wdl/{t}.rtbw'])
print(len(os.listdir('syzygy')), 'Syzygy files')
!python scripts/verify_syzygy.py --syzygy syzygy --sample 200000 tables/*.wmtb

## Build the dataset

Up to 1,000,000 legal positions from each table (all of them for 3-piece tables), one copy of each position, split 90% training / 10% test. 2,000 labels per table are double-checked against Syzygy.

In [ ]:
!python ml/dataset.py --tables tables --out data/dataset.npz --per-table 1000000 --syzygy syzygy --check 2000

## Train the network

10 passes over the training set. Watch the test accuracy climb, and compare it with the baseline (always guessing the most common answer).

In [ ]:
!python ml/train.py --data data/dataset.npz --out model --epochs 10

## Evaluate

1. Accuracy on the test set: overall, per endgame, and per true result.
2. Move choice: does the network's favourite move keep the win (or the draw)?
3. Generalization: 5-piece endgames it never saw (KRPvKR, KQRvKR, KRBvKR, KRNvKR, KQPvKQ), 20,000 random positions each, labelled by Syzygy. Downloads the 5 files first.

In [ ]:
for t in 'KRPvKR KQRvKR KRBvKR KRNvKR KQPvKQ'.split():
    subprocess.run(['wget', '-q', '-nc', '-P', 'syzygy',
                    f'https://tablebase.lichess.ovh/tables/standard/3-4-5-wdl/{t}.rtbw'])
!python ml/evaluate.py --data data/dataset.npz --model model/warpmate_net.pt --tables tables --syzygy syzygy --five KRPvKR KQRvKR KRBvKR KRNvKR KQPvKQ --out model/results.json

## Optional: download the trained network

The weights are small (under 1 MB). Colab deletes everything when the session ends, so download them if you want to keep them.

In [ ]:
!cd model && zip -q ../warpmate_model.zip warpmate_net.pt warpmate_net.npz training_log.json results.json && cd .. && ls -la warpmate_model.zip
from google.colab import files
files.download('warpmate_model.zip')

## Done

Copy the output of every cell from **Tests** onward and paste it back (or download the notebook).